# **Repo Intelligence – deine Agent-Pipeline + BigQuery-MCP, erweitert um alle Auth-Themen**

Dieses Notebook nimmt `agents.ipynb` (Pipeline auf Agent Runtime) und `mcp.ipynb` (BigQuery-MCP auf Cloud Run) und baut daraus **ein** System, das jeden Auth-Weg aus dem Modul einmal benutzt.

**Use Case (neu definiert, damit GitHub + BigQuery Sinn ergeben):** *„Wie gesund ist das GitHub-Repository `owner/name`?“* – Web-News, Statistik aus dem BigQuery-Public-Dataset (Sprachen, Lizenz) und die aktuell offenen Issues aus GitHub, zusammengefasst für einen Engineering Lead.

```
Gemini Enterprise App  ──►  repo_intel (Root, Agent Runtime, deine bestehende Engine)
                              │  Identität: Reasoning-Engine-Service-Agent  (wie bisher)
                              │
      ┌───────────────────────┼─────────────────────────────────────┐
      ▼                       ▼                                     ▼
 web_researcher          bq_stats_agent                        github_agent (RemoteA2aAgent)
 google_search           MCPToolset → ID-Token ──► bq-mcp-analyst    A2A + Access-Token ──► GitHub-Engine
                         (Cloud Run, PRIVAT, run.invoker)             (Agent Runtime, eigene SPIFFE-Identität)
                                                                          │ Auth Manager (API-Key-Provider)
                                                                          ▼
                                                                     GitHub MCP Server (Bearer PAT)
```

| Schritt | Auth-Thema aus dem Modul | Wo im Notebook |
|---|---|---|
| Entwickler → GCP | ADC (`gcloud auth application-default login`) | 0 |
| Auth Manager | API-Key-Provider für GitHub-PAT, `roles/agentidentity.user` | 2, 5 |
| Agent Identity / SPIFFE | GitHub-Engine mit `identity_type=AGENT_IDENTITY`, Trust-Domain `proj-…` (kein Org) | 4, 5 |
| Agent → Agent (A2A) | `A2aAgent` auf Agent Runtime, `RemoteA2aAgent` + Access-Token im Root | 6, 7, 10 |
| Agent → privater Cloud Run | ID-Token aus dem Metadata-Server + `roles/run.invoker` für den Service Agent | 8 |
| Agent Registry | MCP-Server registrieren (Agent-Runtime-Agents sind automatisch drin) | 12 |
| Gemini Enterprise | Root per REST als ADK-Agent registrieren | 13 |
| Least Privilege | Rollen der drei Identitäten prüfen | 14 |

Reihenfolge ist wichtig: erst GitHub-Engine (5–7), dann Root (9–11), denn der Root braucht die Card-URL der GitHub-Engine.


In [1]:
# @title Installs
############################################

# Agent Platform SDK (v1beta1: identity_type, A2aAgent) + ADK mit a2a/mcp/agent-identity + a2a-sdk 1.x
%pip install --upgrade --quiet "google-cloud-aiplatform[agent_engines,adk]>=1.149" "google-adk[a2a,mcp,agent-identity]>=2.7.1" "a2a-sdk>=1.0,<2" google-cloud-trace

%pip show google-adk google-cloud-aiplatform a2a-sdk


Note: you may need to restart the kernel to use updated packages.
Name: google-adk
Version: 2.9.1
Summary: Agent Development Kit
Home-page: https://google.github.io/adk-docs/
Author: 
Author-email: Google LLC <googleapis-packages@google.com>
License: 
Location: /opt/anaconda3/lib/python3.12/site-packages
Requires: aiosqlite, authlib, click, fastapi, google-auth, google-genai, graphviz, httpx, jsonschema, opentelemetry-api, opentelemetry-sdk, packaging, pydantic, python-dotenv, python-multipart, pyyaml, requests, starlette, tenacity, typing-extensions, uvicorn, watchdog, websockets
Required-by: 
---
Name: google-cloud-aiplatform
Version: 2.1.3
Summary: Vertex AI API client library
Home-page: https://github.com/googleapis/python-aiplatform
Author: Google LLC
Author-email: googleapis-packages@google.com
License: Apache 2.0
Location: /opt/anaconda3/lib/python3.12/site-packages
Requires: certifi, docstring_parser, google-api-core, google-auth, google-cloud-bigquery, google-cloud-resource-ma

In [2]:
# @title 0. Environmental Variables & GCP Setup (ADC wie bisher)
############################################

import os
import json
import subprocess
import importlib
import warnings
from getpass import getpass

import requests
import google.auth
import google.auth.transport.requests
import vertexai
from vertexai import types

warnings.filterwarnings("ignore")

PROJECT_ID = "deltorobarba"
PROJECT_NUMBER = "622694106880"
LOCATION = "us-central1"
STAGING_BUCKET = "gs://deltorobarba-agent-staging"

# Bestehende Ressourcen aus agents.ipynb / mcp.ipynb
ROOT_RESOURCE_NAME = "projects/622694106880/locations/us-central1/reasoningEngines/8838312619447156736"
USE_EXISTING_ROOT_ENGINE = True      # False -> neue Engine "repo-intel" statt update der bestehenden
BQ_SERVICE_NAME = "bq-mcp-analyst"
BQ_MCP_URL = "https://bq-mcp-analyst-622694106880.us-central1.run.app"
BQ_SERVICE_ACCOUNT = f"{BQ_SERVICE_NAME}@{PROJECT_ID}.iam.gserviceaccount.com"

GITHUB_USER = "deltorobarba"
DEMO_REPO = "tensorflow/tensorflow"   # steht sicher im BigQuery-Snapshot; dein eigenes Repo evtl. nicht

# Die drei Identitäten, die in diesem Notebook Rechte bekommen:
RE_SERVICE_AGENT = f"service-{PROJECT_NUMBER}@gcp-sa-aiplatform-re.iam.gserviceaccount.com"  # Root-Pipeline
USER_EMAIL = subprocess.check_output(["gcloud", "config", "get-value", "account"], text=True).strip()  # du, lokal
# AGENT_PRINCIPAL (SPIFFE) entsteht in Schritt 4

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

# ⚠️ v1beta1 ist Pflicht für identity_type und A2aAgent
client = vertexai.Client(project=PROJECT_ID, location=LOCATION, http_options=dict(api_version="v1beta1"))

def access_token() -> str:
    # ACCESS-Token deines Users (für Google-APIs, z. B. Discovery Engine) – entspricht gcloud auth print-access-token
    creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
    creds.refresh(google.auth.transport.requests.Request())
    return creds.token

print(f"✅ Lokal als {USER_EMAIL} (ADC) verbunden. Root-Identität deployed: {RE_SERVICE_AGENT}")


✅ Lokal als alexmoheit@gmail.com (ADC) verbunden. Root-Identität deployed: service-622694106880@gcp-sa-aiplatform-re.iam.gserviceaccount.com


In [3]:
# @title 1. APIs aktivieren (⚠️ --> once only!)
############################################

!gcloud services enable aiplatform.googleapis.com run.googleapis.com iam.googleapis.com iamcredentials.googleapis.com sts.googleapis.com agentidentity.googleapis.com agentidentitycredentials.googleapis.com agentregistry.googleapis.com apphub.googleapis.com discoveryengine.googleapis.com cloudtrace.googleapis.com telemetry.googleapis.com logging.googleapis.com --project={PROJECT_ID}


Operation "operations/acat.p2-622694106880-b5907d34-b113-46d9-a50b-9762e8b138ae" finished successfully.


Updates are available for some Google Cloud CLI components.  To install them,
please run:
  $ gcloud components update



In [ ]:
# @title 2. Auth Manager: GitHub-PAT als API-Key-Auth-Provider (idempotent)
############################################

AUTH_PROVIDER_NAME = "github-mcp-auth-provider"
GITHUB_AUTH_PROVIDER_URI = f"projects/{PROJECT_ID}/locations/{LOCATION}/authProviders/{AUTH_PROVIDER_NAME}"

existing = subprocess.run(
    ["gcloud", "agent-identity", "auth-providers", "list", f"--project={PROJECT_ID}",
     f"--location={LOCATION}", "--format=value(name)"],
    capture_output=True, text=True,
).stdout
# Falls deine gcloud-Version den Befehl nur als alpha kennt: ["gcloud", "alpha", "agent-identity", ...]

if AUTH_PROVIDER_NAME in existing:
    print(f"ℹ️ Provider {AUTH_PROVIDER_NAME} existiert bereits (z. B. aus identity.ipynb) – wird wiederverwendet.")
else:
    github_pat = getpass("GitHub Personal Access Token (read-only: repo/read:org bzw. Issues+Metadata Read): ")
    result = subprocess.run(
        ["gcloud", "agent-identity", "auth-providers", "create", AUTH_PROVIDER_NAME,
         f"--project={PROJECT_ID}", f"--location={LOCATION}", f"--api-key={github_pat}"],
        capture_output=True, text=True,
    )
    del github_pat
    print(result.stdout or result.stderr)

!gcloud agent-identity auth-providers list --project={PROJECT_ID} --location={LOCATION}

# Dein User darf den Provider für lokale Tests lesen (Lab: "for local development and testing")
!gcloud agent-identity auth-providers add-iam-policy-binding {AUTH_PROVIDER_NAME} --project={PROJECT_ID} --location={LOCATION} --role="roles/agentidentity.user" --member="user:{USER_EMAIL}" --quiet > /dev/null && echo "✅ user:{USER_EMAIL} darf Provider lesen"
print("Provider-URI:", GITHUB_AUTH_PROVIDER_URI)


## 3. Code als Pakete schreiben

Anders als in `agents.ipynb` liegt der Agent-Code nicht in Notebook-Zellen, sondern in zwei Paketen, die per `extra_packages` mitdeployt werden. Grund: Agent Runtime **pickelt** das Agent-Objekt. Funktionen und Klassen aus einem importierbaren Paket werden dabei nur *per Referenz* gespeichert – so landen weder deine lokalen Credentials noch nicht-picklebare Objekte (httpx-Client mit SSL-Context, MCP-Session-Manager) im Deployment. Die Auth-Objekte initialisieren sich erst in der Runtime – **das ist der Moment, in dem aus „du“ die Agent-Identität wird.**

* `github_agent/` – der A2A-Spezialist mit Auth Manager (Agent Card, Executor, Agent-Builder)
* `repo_intel/` – die Root-Pipeline (`auth.py`: Access-Token/ID-Token, `tools.py`: lazy MCP-Toolset, `callbacks.py`, `agent.py`)


In [ ]:
# @title 3a. Paket github_agent/ schreiben
############################################

from pathlib import Path

FILES = {}

FILES['github_agent/__init__.py'] = r''''''

FILES['github_agent/a2a_config.py'] = r'''"""github_agent/a2a_config.py – Agent Card (das 'Visitenkärtchen' für A2A)."""
from a2a.types import AgentSkill

try:
    from vertexai.agent_engines.templates.a2a import create_agent_card
except ImportError:  # ältere SDK-Versionen
    from vertexai.preview.reasoning_engines.templates.a2a import create_agent_card

github_skill = AgentSkill(
    id="github_issues",
    name="GitHub Issues & Repositories",
    description="Search repositories and list/search open issues via the GitHub MCP server.",
    tags=["github", "issues", "repositories"],
    examples=[
        "List open issues of tensorflow/tensorflow",
        "Which open issues exist in my repositories?",
    ],
)

agent_card = create_agent_card(
    agent_name="GitHub Agent",
    description="Specialist for live GitHub data (issues, repositories) with Agent Identity + Auth Manager.",
    skills=[github_skill],
)
'''

FILES['github_agent/agent.py'] = r'''"""github_agent/agent.py – GitHub-Spezialist, der seinen PAT nicht kennt (Auth Manager).

build_root_agent() wird ERST in der Runtime aufgerufen (vom Executor), damit
McpToolset + Auth-Manager-Registrierung nicht ins Pickle wandern.
"""
import os

from google.adk.agents import LlmAgent
from google.adk.auth.credential_manager import CredentialManager
from google.adk.integrations.agent_identity import GcpAuthProvider, GcpAuthProviderScheme
from google.adk.tools.mcp_tool import McpToolset, StreamableHTTPConnectionParams

GITHUB_TOOL_FILTER = ["search_repositories", "search_issues", "list_issues", "get_issue"]


def build_github_toolset(auth_provider_uri: str, mcp_url: str) -> McpToolset:
    # (1) ADK lernt, den Auth Manager anzusprechen – authentifiziert per ADC = Agent-Identität
    CredentialManager.register_auth_provider(GcpAuthProvider())
    # (2) Welcher Provider vor jedem Tool-Call aufgelöst wird
    auth_scheme = GcpAuthProviderScheme(name=auth_provider_uri)

    toolset = None

    def header_provider(readonly_ctx) -> dict[str, str]:
        """(3) Aufgelöstes Credential -> 'Authorization: Bearer <PAT>'.
        Nötig, weil ein API-Key-Provider den Key als X-GOOG-API-KEY / credentials.token
        liefert, GitHub ihn aber als Bearer-Token erwartet.

        Fail-closed: Ohne Token wird KEIN leerer Header-Dict zurückgegeben (das hiesse:
        anonymer Request -> GitHub antwortet 401 -> sieht nach GitHub-Problem aus),
        sondern hier abgebrochen, wo die eigentliche Ursache noch sichtbar ist.
        """
        if not toolset or not toolset.get_auth_config():
            raise RuntimeError(
                f"Auth Manager nicht aufgeloest: keine auth_config fuer {auth_provider_uri}. "
                "Pruefe roles/agentidentity.user des aufrufenden Principals auf dem Provider."
            )
        auth_config = toolset.get_auth_config()
        cred = None
        if readonly_ctx and hasattr(readonly_ctx, "get_credential"):
            cred = readonly_ctx.get_credential(auth_config.credential_key)
        if not cred:
            cred = auth_config.exchanged_auth_credential
        token = None
        if cred and cred.http:
            if cred.http.credentials and cred.http.credentials.token:
                token = cred.http.credentials.token
            elif cred.http.additional_headers:
                h = cred.http.additional_headers
                token = h.get("X-API-Key") or h.get("X-GOOG-API-KEY")
        if not token:
            raise RuntimeError(
                f"Auth Manager lieferte kein Token fuer {auth_provider_uri} "
                "(weder credentials.token noch X-API-Key / X-GOOG-API-KEY). "
                "Pruefe IAM-Binding und Inhalt des Providers."
            )
        return {"Authorization": f"Bearer {token}"}

    toolset = McpToolset(
        connection_params=StreamableHTTPConnectionParams(url=mcp_url),  # kein headers={...}
        auth_scheme=auth_scheme,
        tool_filter=GITHUB_TOOL_FILTER,
        header_provider=header_provider,
    )
    return toolset


def build_root_agent() -> LlmAgent:
    github_user = os.getenv("GITHUB_USER", "deltorobarba")
    return LlmAgent(
        name="github_agent",
        # gemini-2.5-flash ist regional (us-central1) verfügbar. A2aAgent.set_up() setzt
        # GOOGLE_CLOUD_LOCATION auf die Runtime-Region – die Gemini-3-Familie bräuchte 'global'.
        model=os.getenv("GITHUB_AGENT_MODEL", "gemini-2.5-flash"),
        description="Specialized assistant for GitHub: searches repositories and issues via the GitHub MCP server.",
        instruction=(
            f"You are a GitHub assistant for the user '{github_user}'. "
            "Use the tools to find repositories and issues. For 'open issues of repository owner/name' use "
            "search_issues with the query 'is:issue is:open repo:owner/name'; for the user's own open issues use "
            f"'is:issue is:open user:{github_user}'. Answer concisely and always cite repository name and issue number."
        ),
        tools=[
            build_github_toolset(
                auth_provider_uri=os.environ["GITHUB_AUTH_PROVIDER_URI"],
                mcp_url=os.getenv("GITHUB_MCP_URL", "https://api.githubcopilot.com/mcp/"),
            )
        ],
    )
'''

FILES['github_agent/executor.py'] = r'''"""github_agent/executor.py – Brücke A2A-Protokoll <-> ADK Runner (Vorlage aus der Google-Doku).

Sessions: VertexAiSessionService in der Runtime (GOOGLE_CLOUD_AGENT_ENGINE_ID gesetzt),
InMemorySessionService lokal.
Hinweis: geschrieben für a2a-sdk 1.x (TaskState.TASK_STATE_*). Mit a2a-sdk 0.3.x heißen die
Konstanten TaskState.submitted / .failed und Text-Parts TextPart(text=...).
"""
import logging
import os

import vertexai
from a2a import types as a2a_types
from a2a.server.agent_execution.agent_executor import AgentExecutor
from a2a.server.agent_execution.context import RequestContext
from a2a.server.events.event_queue import EventQueue
from a2a.server.tasks import TaskUpdater
from a2a.types import Part
from google.adk import Runner
from google.adk.artifacts.in_memory_artifact_service import InMemoryArtifactService
from google.adk.memory.in_memory_memory_service import InMemoryMemoryService
from google.adk.sessions import InMemorySessionService, VertexAiSessionService
from google.genai import types as genai_types

logger = logging.getLogger(__name__)


class GithubAgentExecutor(AgentExecutor):
    def __init__(self) -> None:
        self.agent = None
        self.runner = None

    def _init_adk(self) -> None:
        if self.runner is not None:
            return
        from github_agent.agent import build_root_agent  # lazy: erst in der Runtime

        self.agent = build_root_agent()
        engine_id = os.environ.get("GOOGLE_CLOUD_AGENT_ENGINE_ID")
        if engine_id:
            project = os.environ["GOOGLE_CLOUD_PROJECT"]
            location = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")
            vertexai.init(project=project, location=location)
            session_service = VertexAiSessionService(project=project, location=location, agent_engine_id=engine_id)
            app_name = engine_id
        else:
            session_service = InMemorySessionService()
            app_name = self.agent.name
        self.runner = Runner(
            app_name=app_name,
            agent=self.agent,
            artifact_service=InMemoryArtifactService(),
            session_service=session_service,
            memory_service=InMemoryMemoryService(),
        )

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        updater = TaskUpdater(event_queue=event_queue, task_id=context.task_id or "", context_id=context.context_id or "")
        await updater.cancel()

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        self._init_adk()
        if not context.message:
            return
        # user_id kommt (wenn überhaupt) als A2A-Metadatum vom Aufrufer – Vertrauen, kein Beweis
        user_id = (context.message.metadata or {}).get("user_id", "a2a_user")
        updater = TaskUpdater(event_queue, context.task_id, context.context_id)
        task = a2a_types.Task(
            id=context.task_id,
            context_id=context.context_id,
            status=a2a_types.TaskStatus(state=a2a_types.TaskState.TASK_STATE_SUBMITTED),
            history=[context.message],
        )
        await event_queue.enqueue_event(task)
        await updater.start_work()

        query = context.get_user_input()
        content = genai_types.Content(role="user", parts=[genai_types.Part.from_text(text=query)])
        try:
            svc = self.runner.session_service
            session = await svc.get_session(
                app_name=self.runner.app_name, user_id=user_id, session_id=context.context_id
            ) or await svc.create_session(
                app_name=self.runner.app_name, user_id=user_id, session_id=context.context_id
            )
            final_event = None
            async for event in self.runner.run_async(session_id=session.id, user_id=user_id, new_message=content):
                if event.is_final_response():
                    final_event = event
            text = ""
            if final_event and final_event.content and final_event.content.parts:
                text = "".join(p.text for p in final_event.content.parts if getattr(p, "text", None))
            if text:
                await updater.add_artifact([Part(text=text)], name="result", last_chunk=True)
                await updater.complete()
                return
            await updater.update_status(
                a2a_types.TaskState.TASK_STATE_FAILED,
                message=updater.new_agent_message([Part(text="No final text response.")]),
            )
        except Exception:  # Fehler als A2A-Task-Status zurückgeben, nicht crashen
            # Details NUR ins Log (Cloud Logging), nicht über die A2A-Grenze: Exception-Strings
            # aus MCP / Auth Manager enthalten Ressourcennamen, URLs und gelegentlich
            # Header-Fragmente. Der Aufrufer bekommt nur eine generische Meldung.
            logger.exception("github_agent execute() failed (task_id=%s)", context.task_id)
            await updater.update_status(
                a2a_types.TaskState.TASK_STATE_FAILED,
                message=updater.new_agent_message(
                    [Part(text="The GitHub agent could not complete the request. "
                               "See Cloud Logging for details.")]
                ),
            )


def build_executor() -> GithubAgentExecutor:
    """Modul-Level-Funktion -> wird beim Deploy per Referenz gepickelt, nicht per Wert."""
    return GithubAgentExecutor()
'''

for path, content in FILES.items():
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(content)
print("✅ geschrieben:", ", ".join(FILES))

In [ ]:
# @title 3b. Paket repo_intel/ schreiben
############################################

from pathlib import Path

FILES = {}

FILES['repo_intel/__init__.py'] = r''''''

FILES['repo_intel/agent.py'] = r'''"""repo_intel/agent.py – Root-Pipeline "Repo Intelligence" (Weiterentwicklung von agents.ipynb).

Frage an das System: "Wie gesund ist das GitHub-Repository owner/name?"
  parallel_retrieval
    ├── web_researcher   google_search            (wie bisher)
    ├── bq_stats_agent   MCP → PRIVATER Cloud Run  (BigQuery public dataset: Sprachen, Lizenz)
    └── github_agent     A2A → Agent Runtime       (eigener SPIFFE-Agent, GitHub MCP via Auth Manager)
  analyst                load_memory              (wie bisher)
  summarizer                                      (wie bisher)
"""
import os

from google.adk.agents import LlmAgent, ParallelAgent, SequentialAgent
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent
from google.adk.tools import google_search, load_memory

from .auth import GoogleAuthedAsyncClient
from .callbacks import capture_response_to_state
from .tools import LazyBigQueryMcpToolset

MODEL = os.getenv("MODEL", "gemini-3.5-flash")
BQ_MCP_URL = os.environ["BQ_MCP_URL"]                    # https://bq-mcp-analyst-....run.app
GITHUB_AGENT_CARD_URL = os.environ["GITHUB_AGENT_CARD_URL"]  # https://...reasoningEngines/ID/a2a/v1/card

# 1. Web-Recherche (unverändert aus agents.ipynb)
web_agent = LlmAgent(
    name="web_researcher",
    model=MODEL,
    description="Finds latest public news and discussions about a GitHub repository.",
    instruction=(
        "The user names a GitHub repository (owner/name). Search for the latest public news, "
        "releases and discussions about it. Return a short list of findings with URLs."
    ),
    tools=[google_search],
    output_key="web_findings",
)

# 2. BigQuery-Statistik über deinen privaten Cloud-Run-MCP (Identität: Runtime-Service-Agent)
bq_agent = LlmAgent(
    name="bq_stats_agent",
    model=MODEL,
    description="Read-only statistics about a repository from the BigQuery GitHub public dataset.",
    instruction="""You are a data analyst with read-only BigQuery access via the tool `query_public_dataset`.
The user names a GitHub repository (owner/name). Run at most two SMALL queries:
1. Languages:
   SELECT l.name AS language, l.bytes FROM `bigquery-public-data.github_repos.languages`, UNNEST(language) AS l
   WHERE repo_name = '<owner/name>' ORDER BY l.bytes DESC LIMIT 10
2. License:
   SELECT license FROM `bigquery-public-data.github_repos.licenses` WHERE repo_name = '<owner/name>'
NEVER query the tables commits, contents, files or sample_* (terabytes; cost guard).
If a query returns no rows, state that the repository is not in the public snapshot.
Report the results as a compact list.""",
    tools=[LazyBigQueryMcpToolset(BQ_MCP_URL)],
    output_key="bq_findings",
)

# 3. GitHub-Spezialist als REMOTE Agent über A2A (Identität des Aufrufers: ADC-Access-Token)
github_agent = RemoteA2aAgent(
    name="github_agent",
    description="Live GitHub data: open issues and repository search via the GitHub MCP server.",
    agent_card=GITHUB_AGENT_CARD_URL,
    httpx_client=GoogleAuthedAsyncClient(),            # Agent Runtime ist immer authentifiziert
    after_agent_callback=capture_response_to_state("github_findings"),
)

parallel_retrieval = ParallelAgent(
    name="parallel_retrieval",
    sub_agents=[web_agent, bq_agent, github_agent],
)

# 4. Analyst mit Memory (unverändert im Prinzip)
analyst = LlmAgent(
    name="analyst",
    model=MODEL,
    instruction="""Check your memory for user-specific preferences.
Combine three sources about the repository:
- Web findings: {web_findings?}
- BigQuery statistics (languages, license): {bq_findings?}
- Live GitHub issues: {github_findings?}
Assess the repository's health: activity, language mix, license, open-issue themes, notable news.
Mention explicitly if a source returned nothing.""",
    tools=[load_memory],
    output_key="final_analysis",
)

# 5. Summarizer (unverändert)
summarizer = LlmAgent(
    name="summarizer",
    model=MODEL,
    instruction=(
        "You are an Engineering Intelligence Analyst. Summarize {final_analysis} into 5 punchy "
        "bullets for an engineering lead. Cite sources (URL / issue number / table). Max 2000 characters."
    ),
    output_key="summary",
)

root_agent = SequentialAgent(
    name="repo_intel",
    sub_agents=[parallel_retrieval, analyst, summarizer],
)
'''

FILES['repo_intel/auth.py'] = r'''"""repo_intel/auth.py – Auth-Bausteine für den Root-Agent (Agent → Agent, Agent → Cloud Run).

Zwei Regeln, die hier eingebaut sind:
  1. Credentials werden NIE beim Import/Deploy erzeugt, sondern erst beim ersten Request
     in der Runtime. Sonst würden deine User-Credentials (Refresh Token!) mit in das
     gepickelte Deployment wandern – und die Runtime-Identität wäre nie "der Agent".
  2. httpx-Clients enthalten einen SSL-Context und sind nicht picklebar. __reduce__ sorgt
     dafür, dass beim Deploy nur "erzeuge mich neu" gespeichert wird.
"""
import threading
import time

import google.auth
import google.auth.transport.requests
import google.oauth2.id_token
import httpx

_SCOPES = ["https://www.googleapis.com/auth/cloud-platform"]


class GoogleADCAuth(httpx.Auth):
    """OAuth-ACCESS-Token aus ADC an jede Anfrage hängen (Ziel: Agent Runtime / Google APIs).

    Lokal = dein User, deployed = Runtime-Service-Agent. Gleicher Code, andere Identität.
    """

    def __init__(self) -> None:
        self._creds = None
        self._request = None
        self._lock = threading.Lock()

    def _token(self) -> str:
        # ParallelAgent laesst web_researcher / bq_stats_agent / github_agent gleichzeitig
        # laufen. Ohne Lock koennten zwei Threads parallel google.auth.default() bzw.
        # creds.refresh() auf demselben Objekt ausfuehren - beides ist nicht thread-safe.
        with self._lock:
            if self._creds is None:
                self._creds, _ = google.auth.default(scopes=_SCOPES)
                self._request = google.auth.transport.requests.Request()
            if not self._creds.valid:
                self._creds.refresh(self._request)
            return self._creds.token

    def auth_flow(self, request: httpx.Request):
        request.headers["Authorization"] = f"Bearer {self._token()}"
        yield request

    # Beim Pickling keine Credentials mitnehmen (und kein Lock - nicht picklebar)
    def __getstate__(self):
        return {}

    def __setstate__(self, state):
        self._creds = None
        self._request = None
        self._lock = threading.Lock()


class GoogleAuthedAsyncClient(httpx.AsyncClient):
    """httpx.AsyncClient mit ADC-Auth, der sich beim Unpickling in der Runtime neu erzeugt."""

    def __init__(self, timeout: float = 180.0) -> None:
        super().__init__(auth=GoogleADCAuth(), timeout=timeout)

    def __reduce__(self):
        return (GoogleAuthedAsyncClient, ())


_id_cache: dict[str, tuple[str, float]] = {}


def cloud_run_id_token(audience: str) -> str:
    """OIDC-ID-Token mit aud = Cloud-Run-URL (Ziel: PRIVATER Cloud-Run-Dienst), 1 h gecacht.

    Deployed: der Metadata-Server mintet das Token für die Runtime-Identität
              (Service Agent / Service Account) – google.oauth2.id_token.fetch_id_token.
    Lokal:    User-ADC hat keinen Metadata-Server -> Fallback auf das gcloud-User-ID-Token
              (creds.id_token), genau wie in mcp.ipynb.
    """
    now = time.time()
    cached = _id_cache.get(audience)
    if cached and now < cached[1] - 300:
        return cached[0]
    request = google.auth.transport.requests.Request()
    try:
        token = google.oauth2.id_token.fetch_id_token(request, audience)
    except Exception:
        creds, _ = google.auth.default()
        creds.refresh(request)
        token = getattr(creds, "id_token", None)
        if not token:
            raise RuntimeError(
                "Kein ID-Token verfügbar: weder Metadata-Server (deployed) noch User-ADC (lokal)."
            )
    _id_cache[audience] = (token, now + 3600)
    return token
'''

FILES['repo_intel/callbacks.py'] = r'''"""repo_intel/callbacks.py – Antwort eines RemoteA2aAgent in den Session-State kopieren.

Ein RemoteA2aAgent hat kein output_key (Lab 1, Task 5): sein Ergebnis landet nicht
automatisch im State. Dieser after_agent_callback holt den letzten Text-Event des
Agents aus der Session und speichert ihn unter state[state_key], damit der Analyst
ihn per {state_key?} lesen kann.
"""


def capture_response_to_state(state_key: str):
    async def _after_agent_callback(callback_context):
        session = getattr(callback_context, "session", None)
        if session is None:  # ältere ADK-Versionen
            session = callback_context._invocation_context.session
        for event in reversed(session.events):
            if event.author != callback_context.agent_name or not event.content:
                continue
            parts = event.content.parts or []
            text = "".join(p.text for p in parts if getattr(p, "text", None))
            if text:
                callback_context.state[state_key] = text
                break
        return None

    return _after_agent_callback
'''

FILES['repo_intel/tools.py'] = r'''"""repo_intel/tools.py – BigQuery-MCP auf privatem Cloud Run, aus dem Root-Agent heraus."""
from typing import Optional

from google.adk.tools.base_toolset import BaseToolset
from google.adk.tools.mcp_tool import McpToolset, SseConnectionParams

from .auth import cloud_run_id_token


class LazyBigQueryMcpToolset(BaseToolset):
    """Wrapper um McpToolset, der die MCP-Verbindung erst in der Runtime aufbaut.

    Warum? Ein McpToolset hält Session-Manager/Locks und wäre im Pickle-Deploy ein
    Risiko. Dieser Wrapper enthält nur die URL; beim ersten get_tools() entsteht das
    echte Toolset – lokal mit deinem ID-Token, deployed mit dem des Service Agents.
    """

    def __init__(self, cloud_run_url: str) -> None:
        super().__init__()
        self._url = cloud_run_url.rstrip("/")
        self._inner: Optional[McpToolset] = None

    def _get_inner(self) -> McpToolset:
        if self._inner is None:
            url = self._url
            self._inner = McpToolset(
                connection_params=SseConnectionParams(url=f"{url}/sse"),
                # header_provider statt headers={...}: Token wird pro Request frisch geliefert
                header_provider=lambda ctx: {"Authorization": f"Bearer {cloud_run_id_token(url)}"},
            )
        return self._inner

    async def get_tools(self, readonly_context=None):
        return await self._get_inner().get_tools(readonly_context)

    async def close(self) -> None:
        if self._inner is not None:
            await self._inner.close()

    # Nur die URL wird gepickelt, nie das innere Toolset
    def __getstate__(self):
        return {"_url": self._url}

    def __setstate__(self, state):
        self.__init__(state["_url"])
'''

for path, content in FILES.items():
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(content)
print("✅ geschrieben:", ", ".join(FILES))

In [ ]:
# @title 4. GitHub-Engine, Deploy Schritt 1: nur Identität (SPIFFE) anlegen (⚠️ --> once only!)
############################################
# Zwei-Schritt-Muster wie im Lab-Skript deploy_with_identity.sh:
# Engine mit identity_type=AGENT_IDENTITY ohne Code -> Principal lesen -> IAM -> Code per update()

gh_engine = client.agent_engines.create(
    config={"display_name": "github-a2a-agent", "identity_type": types.IdentityType.AGENT_IDENTITY},
)
GH_ENGINE_NAME = gh_engine.api_resource.name
GH_ENGINE_ID = GH_ENGINE_NAME.split("/")[-1]
EFFECTIVE_IDENTITY = gh_engine.api_resource.spec.effective_identity
AGENT_PRINCIPAL = f"principal://{EFFECTIVE_IDENTITY}"

print("Engine:    ", GH_ENGINE_NAME)
print("Identität: ", EFFECTIVE_IDENTITY)
print("IAM-Member:", AGENT_PRINCIPAL)
# Ohne Organisation (privates Projekt) erwartest du:
#   agents.global.proj-622694106880.system.id.goog/resources/aiplatform/projects/622694106880/locations/us-central1/reasoningEngines/<ID>
# Lab (mit Org): agents.global.org-<ORG_ID>.system.id.goog/...


In [ ]:
# @title 5. IAM für den SPIFFE-Principal: Provider lesen + Modell/Sessions + Telemetrie
############################################

# (a) Auf dem Auth Provider – gleiche Rolle wie für deinen User, anderer Member-String
!gcloud agent-identity auth-providers add-iam-policy-binding {AUTH_PROVIDER_NAME} --project={PROJECT_ID} --location={LOCATION} --role="roles/agentidentity.user" --member="{AGENT_PRINCIPAL}" --quiet > /dev/null && echo "✅ agentidentity.user auf Provider"

# (b) Im Projekt
for role in ["roles/aiplatform.user", "roles/serviceusage.serviceUsageConsumer",
             "roles/logging.logWriter", "roles/cloudtrace.agent"]:
    !gcloud projects add-iam-policy-binding {PROJECT_ID} --member="{AGENT_PRINCIPAL}" --role="{role}" --condition=None --quiet > /dev/null && echo "✅ {role}"

# (c) Der Root (Service Agent) muss die GitHub-Engine über A2A aufrufen dürfen:
#     roles/aiplatform.user enthält aiplatform.reasoningEngines.query – laut agents.ipynb bereits vergeben; sicherstellen:
!gcloud projects add-iam-policy-binding {PROJECT_ID} --member="serviceAccount:{RE_SERVICE_AGENT}" --role="roles/aiplatform.user" --condition=None --quiet > /dev/null && echo "✅ Service Agent darf Reasoning Engines aufrufen"

print("Hinweis: IAM-Propagation dauert 1–2 Minuten.")


In [ ]:
# @title 6. GitHub-Engine, Deploy Schritt 2: A2aAgent-Code auf die Engine laden (update)
############################################

try:
    from vertexai.agent_engines.templates.a2a import A2aAgent
except ImportError:
    from vertexai.preview.reasoning_engines import A2aAgent

from github_agent.a2a_config import agent_card       # Agent Card (a2a.types) – picklebar
from github_agent.executor import build_executor    # Modul-Funktion -> per Referenz gepickelt

a2a_agent = A2aAgent(agent_card=agent_card, agent_executor_builder=build_executor)

gh_env = {
    "GITHUB_AUTH_PROVIDER_URI": GITHUB_AUTH_PROVIDER_URI,     # nur ein Ressourcenname, kein Secret
    "GITHUB_MCP_URL": "https://api.githubcopilot.com/mcp/",
    "GITHUB_USER": GITHUB_USER,
    "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "true",
    "OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT": "true",
}

gh_engine = client.agent_engines.update(
    name=GH_ENGINE_NAME,                       # Identität aus Schritt 4 bleibt unangetastet
    agent=a2a_agent,
    config={
        "display_name": "github-a2a-agent",
        "description": agent_card.description,
        "staging_bucket": STAGING_BUCKET,
        "requirements": [
            "google-cloud-aiplatform[agent_engines,adk]>=1.149",
            "google-adk[a2a,mcp,agent-identity]>=2.7.1",
            "a2a-sdk>=1.0,<2",
            "cloudpickle", "pydantic",
        ],
        "extra_packages": ["./github_agent"],  # Code wird als Paket mitgeliefert, nicht gepickelt
        "env_vars": gh_env,
    },
)
print("✅ deployed – Identität:", gh_engine.api_resource.spec.effective_identity)


In [ ]:
# @title 7. Test GitHub-Engine ====> Agent Card holen + Aufruf über RemoteA2aAgent (lokal)
############################################
# Agent Runtime liefert KEINE öffentliche Card – nur die authentifizierte (Access-Token nötig).

gh_engine = client.agent_engines.get(name=GH_ENGINE_NAME)
card = await gh_engine.handle_authenticated_agent_card()
GITHUB_AGENT_CARD_URL = f"{card.url}/v1/card"          # das ist die URL, die der Root braucht
print("Agent:   ", card.name)
print("A2A URL: ", card.url)
print("Card URL:", GITHUB_AGENT_CARD_URL)

# Direkt-Test genau so, wie es der Root später macht: RemoteA2aAgent + Access-Token (jetzt: dein ADC-User)
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent
from google.adk.runners import InMemoryRunner
from google.genai import types as genai_types
from repo_intel.auth import GoogleAuthedAsyncClient

remote_github = RemoteA2aAgent(
    name="github_agent",
    description="GitHub specialist over A2A",
    agent_card=GITHUB_AGENT_CARD_URL,
    httpx_client=GoogleAuthedAsyncClient(),
)

async def ask_agent(agent, question: str, app_name: str):
    runner = InMemoryRunner(agent=agent, app_name=app_name)
    sess = await runner.session_service.create_session(app_name=app_name, user_id=GITHUB_USER)
    msg = genai_types.Content(role="user", parts=[genai_types.Part.from_text(text=question)])
    async for event in runner.run_async(user_id=GITHUB_USER, session_id=sess.id, new_message=msg):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if getattr(part, "text", None):
                    print(f"[{event.author}] {part.text}")

await ask_agent(remote_github, f"List the 5 newest open issues of {DEMO_REPO}.", "a2a_test")
# Was hier passiert: dein Access-Token -> Agent Runtime (401 ohne Token / 403 ohne aiplatform.user)
# -> Executor -> ADK -> SPIFFE-Identität -> Auth Manager -> PAT -> Bearer -> GitHub MCP.


In [ ]:
# @title 8. Privater Cloud-Run-MCP: Service Agent darf aufrufen + lokaler Test des Toolsets
############################################
# In mcp.ipynb darf bisher nur user:{USER_EMAIL} den Dienst aufrufen. Der deployte Root ruft ihn
# als Service Agent auf -> braucht roles/run.invoker und ein ID-Token mit aud = Service-URL.

!gcloud run services add-iam-policy-binding {BQ_SERVICE_NAME} --region={LOCATION} --project={PROJECT_ID} --member="serviceAccount:{RE_SERVICE_AGENT}" --role="roles/run.invoker" --quiet > /dev/null && echo "✅ Service Agent ist run.invoker auf {BQ_SERVICE_NAME}"

# Der MCP-Server selbst braucht BigQuery-Rechte (in mcp.ipynb nicht sichtbar) – sicherstellen:
!gcloud projects add-iam-policy-binding {PROJECT_ID} --member="serviceAccount:{BQ_SERVICE_ACCOUNT}" --role="roles/bigquery.jobUser" --condition=None --quiet > /dev/null && echo "✅ {BQ_SERVICE_ACCOUNT}: bigquery.jobUser"

# Lokaler Test: gleicher Code wie deployed – hier greift der Fallback auf dein User-ID-Token
from repo_intel.tools import LazyBigQueryMcpToolset
bq_tools = LazyBigQueryMcpToolset(BQ_MCP_URL)
tools = await bq_tools.get_tools()
print("✅ MCP erreichbar, Tools:", [t.name for t in tools])
await bq_tools.close()


In [ ]:
# @title 9. Root-Pipeline lokal testen (Playground-Äquivalent: läuft mit DEINEN Credentials)
############################################

os.environ["BQ_MCP_URL"] = BQ_MCP_URL
os.environ["GITHUB_AGENT_CARD_URL"] = GITHUB_AGENT_CARD_URL
os.environ["MODEL"] = "gemini-3.5-flash"
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"      # Gemini-3-Familie: global endpoint (wie in agents.ipynb)

import repo_intel.agent as ri
importlib.reload(ri)

await ask_agent(ri.root_agent, f"Analyze the health of the GitHub repository {DEMO_REPO}.", "repo_intel_local")

os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
# Lokal funktioniert alles mit deinen Rechten. Nach dem Deploy (Schritt 10) ruft der Root als
# Service Agent auf – deshalb die Grants in Schritt 5(c) und 8. Fehlen sie: 403, nicht 401.


In [ ]:
# @title 10. Root-Pipeline deployen (update deiner bestehenden Engine, wie in agents.ipynb)
############################################

from vertexai.agent_engines import AdkApp

root_env = {
    "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "true",
    "OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT": "true",
    "GOOGLE_CLOUD_LOCATION": "global",                # wie bisher: Gemini 3.5 auf global
    "MODEL": "gemini-3.5-flash",
    "BQ_MCP_URL": BQ_MCP_URL,                         # privater Cloud Run – kein Secret, nur URL
    "GITHUB_AGENT_CARD_URL": GITHUB_AGENT_CARD_URL,   # A2A-Ziel – kein Secret, nur URL
}

root_config = {
    "display_name": "Repo Intelligence (A2A + MCP + Auth Manager)",
    "staging_bucket": STAGING_BUCKET,
    "requirements": [
        "google-cloud-aiplatform[agent_engines,adk]>=1.149",
        "google-adk[a2a,mcp]>=2.7.1",
        "a2a-sdk>=1.0,<2",
        "google-cloud-trace",
    ],
    "extra_packages": ["./repo_intel"],
    "env_vars": root_env,
    "context_spec": {   # Memory Bank wie bisher
        "memory_bank_config": {
            "generation_config": {
                "model": f"projects/{PROJECT_ID}/locations/global/publishers/google/models/gemini-3.5-flash"
            }
        }
    },
}

adk_app = AdkApp(agent=ri.root_agent, enable_tracing=True)

if USE_EXISTING_ROOT_ENGINE:
    root_engine = client.agent_engines.update(name=ROOT_RESOURCE_NAME, agent=adk_app, config=root_config)
else:
    root_engine = client.agent_engines.create(agent=adk_app, config=root_config)

ROOT_RESOURCE_NAME = root_engine.api_resource.name
print("✅ Root deployed:", ROOT_RESOURCE_NAME)
print("   Identität:", root_engine.api_resource.spec.effective_identity)   # = Service Agent (kein AGENT_IDENTITY)


In [ ]:
# @title 11. Test Agent ====> Single Turn Query (wie in agents.ipynb)
############################################

remote_app = client.agent_engines.get(name=ROOT_RESOURCE_NAME)

async def run_remote_test(message: str, user_id: str = GITHUB_USER):
    raw_session = await remote_app.async_create_session(user_id=user_id)
    session_id = raw_session.get("id")
    print(f"✅ Managed Session ID: {session_id}")
    async for event in remote_app.async_stream_query(user_id=user_id, session_id=session_id, message=message):
        author = event.get("author")
        if author in ("bq_stats_agent", "github_agent", "summarizer") and "content" in event:
            for part in event["content"].get("parts", []):
                if "text" in part:
                    print(f"\n--- [{author}] ---\n{part['text']}")
    # Memory-Extraktion wie bisher
    current_session = await remote_app.async_get_session(user_id=user_id, session_id=session_id)
    if current_session.get("events"):
        await remote_app.async_add_session_to_memory(session=current_session)
        print("\n[Done] Memory extraction triggered.")

await run_remote_test(f"I lead the platform team. Analyze the health of the GitHub repository {DEMO_REPO}.")

# Beweisführung (optional):
#  - run.invoker des Service Agents entfernen  -> bq_stats_agent scheitert (403 vom Cloud Run), Rest läuft
#  - aiplatform.user des Service Agents entfernen -> github_agent scheitert (403 von Agent Runtime)
#  - agentidentity.user des SPIFFE-Principals entfernen -> GitHub-Engine antwortet, aber ohne Token (Provider-Fehler)


In [ ]:
# @title 12. Agent Registry: MCP-Server registrieren, Agents auflisten
############################################
# Agent-Runtime-Agents (Root, GitHub-Engine) werden automatisch katalogisiert.
# Der Cloud-Run-MCP-Server nicht -> registrieren, damit er in Agent Topology / SCC AI Protection erscheint.

!gcloud alpha agent-registry services create bq-mcp --project={PROJECT_ID} --location={LOCATION} --display-name="bq-mcp (BigQuery MCP on Cloud Run)" --mcp-server-spec-type=no-spec --interfaces="url={BQ_MCP_URL}/sse,protocolBinding=JSONRPC"
# (Registrierung ist idempotent; falls die CLI 'protocol=' statt 'protocolBinding=' erwartet, anpassen.)

print("\n== Agents ==")
!gcloud alpha agent-registry agents list --location={LOCATION} --project={PROJECT_ID} --format="table(displayName)"
print("\n== MCP-Server ==")
!gcloud alpha agent-registry mcp-servers list --location={LOCATION} --project={PROJECT_ID}
# Console: Agent Platform > Govern > Agent Registry; Topologie-Graph füllt sich aus Traces (Batch, bis zu Stunden).


In [ ]:
# @title 13. Gemini Enterprise: Root als ADK-Agent registrieren (REST, adkAgentDefinition)
############################################

DISCOVERY = "https://discoveryengine.googleapis.com/v1alpha"      # App-Location 'global'; für 'us': https://us-discoveryengine...
HEADERS = {"Authorization": f"Bearer {access_token()}", "Content-Type": "application/json",
           "X-Goog-User-Project": PROJECT_ID}                         # Quota-Projekt für User-Credentials

# 13a. Deine Gemini-Enterprise-App (Engine) finden
engines = requests.get(
    f"{DISCOVERY}/projects/{PROJECT_ID}/locations/global/collections/default_collection/engines",
    headers=HEADERS).json()
for e in engines.get("engines", []):
    print(e["name"].split("/")[-1], "|", e.get("displayName"))

GE_APP_ID = "PASTE_YOUR_APP_ID"        # <-- App-ID aus der Liste oben eintragen

# 13b. Registrieren
body = {
    "displayName": "Repo Intelligence",
    "description": ("Analyzes the health of a GitHub repository: web news, BigQuery statistics "
                    "(languages, license) and live open issues from GitHub."),
    "adkAgentDefinition": {
        "provisionedReasoningEngine": {"reasoningEngine": ROOT_RESOURCE_NAME}
    },
}
resp = requests.post(
    f"{DISCOVERY}/projects/{PROJECT_ID}/locations/global/collections/default_collection/engines/{GE_APP_ID}"
    f"/assistants/default_assistant/agents",
    headers=HEADERS, json=body)
print(resp.status_code, json.dumps(resp.json(), indent=2)[:800])

# Danach in der Console: Gemini Enterprise > App > Agents > "Repo Intelligence" > User permissions > Rolle "Agent User".
# Gemini Enterprise übergibt dem ADK-Agent die E-Mail des Nutzers -> Memory Bank ist pro Nutzer gescoped.


In [ ]:
# @title 14. Least Privilege: Rollen der drei Identitäten prüfen
############################################

def roles_of(member: str):
    out = subprocess.run(
        ["gcloud", "projects", "get-iam-policy", PROJECT_ID, "--flatten=bindings[].members",
         f"--filter=bindings.members:{member}", "--format=value(bindings.role)"],
        capture_output=True, text=True).stdout.split()
    print(f"\n{member}\n  " + "\n  ".join(out or ["(keine Projekt-Rollen)"]))

roles_of(f"serviceAccount:{RE_SERVICE_AGENT}")   # Root: aiplatform.user, discoveryengine.viewer, cloudtrace.agent – kein editor!
roles_of(f"serviceAccount:{BQ_SERVICE_ACCOUNT}") # MCP-Server: bigquery.jobUser (+ ggf. dataViewer)
roles_of(AGENT_PRINCIPAL)                        # GitHub-Engine: aiplatform.user, serviceUsageConsumer, logWriter, cloudtrace.agent

print("\nCloud-Run-Invoker auf", BQ_SERVICE_NAME)
!gcloud run services get-iam-policy {BQ_SERVICE_NAME} --region={LOCATION} --project={PROJECT_ID} --format="value(bindings.members)"
# Tiefer: IAM & Admin > Policy Analyzer (Principal = eine der drei Identitäten) bzw. IAM Recommender nach ~90 Tagen.


## Was jetzt abgedeckt ist – und was bewusst fehlt

| Thema | Status |
|---|---|
| ADC lokal vs. Runtime-Identität deployed | ✅ Schritt 0 / 9 vs. 10 |
| Access-Token → Agent Runtime (A2A) mit `roles/aiplatform.user` | ✅ 5c, 7, 10 |
| ID-Token → privater Cloud Run mit `roles/run.invoker` | ✅ 8 |
| Agent Identity (SPIFFE), Trust-Domain ohne Org | ✅ 4, 5 |
| Auth Manager (API-Key-Provider, `header_provider`) | ✅ 2, 3a |
| Agent → Agent über A2A (`A2aAgent` Server, `RemoteA2aAgent` Client, Card-URL) | ✅ 6, 7, 10 |
| State-Übernahme eines Remote-Agents (`capture_response_to_state`) | ✅ 3b |
| Agent Registry (MCP-Server) | ✅ 12 |
| Gemini Enterprise (REST-Registrierung) | ✅ 13 |
| Least Privilege | ✅ 14 |
| Memory Bank + Tracing | ✅ unverändert übernommen |

Bewusst **nicht** drin (kein Ziel-System bzw. Org-Level nötig):

* **2LO / 3LO-Provider** – braucht ein OAuth-fähiges Zielsystem (Salesforce, Jira). Der Code-Pfad wäre `AuthenticatedFunctionTool` + `AuthConfig` aus Modul 1, Abschnitt 6.6.
* **Agent Gateway** (mTLS + DPoP, `roles/iap.egressor`, CEL) – Produktions-Fleet-Governance; die Codelabs dazu sind in der Doku verlinkt.
* **IAP vor dem MCP-Server** – Alternative zu `run.invoker`; Doku „Authenticate your MCP server with IAP“.
* **Cloud-Run-*Agent* über A2A** (Lab: LangGraph auf Cloud Run) – hier ist Cloud Run nur MCP-Server. Wenn du das Muster willst: `to_a2a(root_agent)` aus ADK + `gcloud run deploy`, Root ruft ohne Token (public) oder mit ID-Token (privat).
* **SCC AI Protection / Attack Path** – Org-Level.

### Troubleshooting

| Symptom | Ursache / Fix |
|---|---|
| `identity_type` / `A2aAgent` unbekannt | Client ohne `http_options=dict(api_version="v1beta1")` bzw. altes SDK |
| 401 beim Card-Abruf | Agent Runtime liefert nur die *authentifizierte* Card; Access-Token fehlt |
| 403 nach Root-Deploy, lokal ging es | Service Agent hat `aiplatform.user` (A2A) oder `run.invoker` (Cloud Run) nicht – Schritt 5c / 8 |
| `401 Context-Aware Access requirements are not met` | Token der SPIFFE-Identität außerhalb der Runtime verwendet – gewollt |
| Provider-/`CONNECTOR_AUTH_TOKEN_EXCHANGE_ERROR` statt GitHub-Fehler | Auth Manager löst *vor* dem Tool-Call auf: IAM-Binding oder Provider prüfen |
| `cannot pickle …` beim Deploy | Ein Objekt wurde per Wert gepickelt: Code muss im `extra_packages`-Paket liegen, Clients lazy |
| BigQuery „not in snapshot“ | `github_repos` ist ein älterer Snapshot – `tensorflow/tensorflow` ist sicher drin |
| a2a-sdk-Attributfehler (`TaskState`) | Version-Mismatch 0.3.x vs 1.x – Pins in Schritt 6 und lokal angleichen |


In [ ]:
# @title (Optional) Aufräumen
############################################
# client.agent_engines.get(name=GH_ENGINE_NAME).delete(force=True)
# !gcloud agent-identity auth-providers delete {AUTH_PROVIDER_NAME} --project={PROJECT_ID} --location={LOCATION}
# !gcloud alpha agent-registry services delete bq-mcp --project={PROJECT_ID} --location={LOCATION}
# Achtung: IAM-Bindings auf gelöschte principal://-Identitäten bleiben bestehen – manuell entfernen.
